In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict, OrderedDict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (
    LassoCV,
    ElasticNetCV,
    LogisticRegressionCV,
    LinearRegression,
    LogisticRegression
)
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, roc_auc_score, accuracy_score
import warnings

warnings.filterwarnings("ignore")

In [2]:
# =========================
# USER INPUTS
# =========================

id_col = "admission_id"          # replace with your subject/stay identifier
time_col = "t0"        # your discrete time variable
max_lag = 1            # start with lag1 only; later you can try lag2
random_state = 42

covnames = [
    'vent_mode__last__last_12h',
    'vent_mode__hours_since_last__last_12h',
    
    'pco2_arterial__mean__last_12h',
    'po2_arterial__mean__last_12h',
    'fio2__last__last_12h',
    'o2_saturation__mean__last_12h',
    'respiratory_rate_measured__mean__last_12h',
    
    'glasgow_coma_scale_total__last__last_12h',

    'lactate__last__last_12h',
    'fluid_out_urine__mean__last_12h',
    'ureum__last__last_12h',
    'creatinine__last__last_12h',

    'arterial_blood_pressure_mean__mean__last_12h',
    'heart_rate__mean__last_12h',
    'hemoglobin__last__last_12h',
    
    'temperature__mean__last_12h',
    'activated_partial_thromboplastin_time__last__last_12h',
    'bicarbonate_arterial__last__last_12h'
]

# Add your baseline/static covariates here
baseline_cols = [
    "age", "sex", "origin"
    #"weight", "height", "unit_type"
]

# Optional: variables you want to always force into the model if available
# Example: always include time and own lag
force_include_time = True
force_include_own_lag = True

# Stability selection settings
n_bootstrap = 50
subsample_frac = 0.7
selection_threshold = 0.6   # keep vars selected in at least 60% of resamples

# Refit settings
min_selected_predictors = 1

In [3]:
# =========================
# TARGET TYPE MAP
# supported:
# - "continuous"
# - "binary"
# - "categorical"
# =========================

target_type_map = {
    'vent_mode__last__last_12h': 'categorical',
    'vent_mode__hours_since_last__last_12h': 'continuous',
    
    'pco2_arterial__mean__last_12h': 'continuous',
    'po2_arterial__mean__last_12h': 'continuous',
    'fio2__last__last_12h': 'categorical',
    'o2_saturation__mean__last_12h': 'continuous',
    'respiratory_rate_measured__mean__last_12h': 'continuous',
    
    'glasgow_coma_scale_total__last__last_12h': 'categorical',

    'lactate__last__last_12h': 'continuous',
    'fluid_out_urine__mean__last_12h': 'continuous',
    'ureum__last__last_12h': 'continuous',
    'creatinine__last__last_12h': 'continuous',

    'arterial_blood_pressure_mean__mean__last_12h': 'continuous',
    'heart_rate__mean__last_12h': 'continuous',
    'hemoglobin__last__last_12h': 'continuous',
    
    'temperature__mean__last_12h': 'continuous',
    'activated_partial_thromboplastin_time__last__last_12h': 'continuous',
    'bicarbonate_arterial__last__last_12h': 'continuous'
}

In [4]:
# =========================
# MANUAL BLOCK ORDERING
# =========================

block_order = OrderedDict({
    "support_process": [
        'vent_mode__last__last_12h',
        'fio2__last__last_12h',
        'vent_mode__hours_since_last__last_12h',
    ],
    "respiratory_state": [
        'respiratory_rate_measured__mean__last_12h',
        'o2_saturation__mean__last_12h',
        'pco2_arterial__mean__last_12h',
        'po2_arterial__mean__last_12h',
    ],
    "neurologic": [
        'glasgow_coma_scale_total__last__last_12h',
    ],
    "hemodynamic_perfursion": [
        'arterial_blood_pressure_mean__mean__last_12h',
        'heart_rate__mean__last_12h',
        'temperature__mean__last_12h',
        'lactate__last__last_12h',
    ],
    "renal_fluid": [
        'fluid_out_urine__mean__last_12h',
        'creatinine__last__last_12h',
        'ureum__last__last_12h',
    ],
    "blood_coag_acidbase": [
        'hemoglobin__last__last_12h',
        'activated_partial_thromboplastin_time__last__last_12h',
        'bicarbonate_arterial__last__last_12h',
    ]
})

In [5]:
def get_block_index_map(block_order):
    block_index = {}
    var_to_block = {}
    ordered_vars = []
    for i, (block_name, vars_in_block) in enumerate(block_order.items()):
        for v in vars_in_block:
            block_index[v] = i
            var_to_block[v] = block_name
            ordered_vars.append(v)
    return block_index, var_to_block, ordered_vars

block_index_map, var_to_block_map, ordered_covs = get_block_index_map(block_order)

assert set(ordered_covs) == set(covnames), "block_order must contain exactly the same variables as covnames"

In [6]:
def add_lag_features(df, id_col, time_col, variables, max_lag=1):
    df = df.sort_values([id_col, time_col]).copy()
    for lag in range(1, max_lag + 1):
        for v in variables:
            df[f"lag{lag}_{v}"] = df.groupby(id_col)[v].shift(lag)
    return df

In [7]:
def get_allowed_same_time_predictors(target, covnames, block_index_map):
    """
    Same-time predictors allowed only from earlier blocks.
    """
    target_block_idx = block_index_map[target]
    allowed = [v for v in covnames if block_index_map[v] < target_block_idx]
    return allowed


def get_allowed_predictors(
    target,
    covnames,
    baseline_cols,
    time_col,
    max_lag,
    block_index_map,
    force_include_time=True,
    force_include_own_lag=True
):
    predictors = []

    # baseline covariates
    predictors.extend(baseline_cols)

    # time
    if force_include_time:
        predictors.append(time_col)

    # lags of all covariates
    for lag in range(1, max_lag + 1):
        predictors.extend([f"lag{lag}_{v}" for v in covnames])

    # same-time predictors only from earlier blocks
    predictors.extend(get_allowed_same_time_predictors(target, covnames, block_index_map))

    # remove self at same time if somehow present
    predictors = [p for p in predictors if p != target]

    # optional: ensure own lag is present
    if force_include_own_lag:
        own_lag = f"lag1_{target}"
        if own_lag not in predictors and max_lag >= 1:
            predictors.append(own_lag)

    # unique preserving order
    predictors = list(dict.fromkeys(predictors))
    return predictors

In [8]:
def infer_feature_types(df, feature_cols):
    numeric_cols = []
    categorical_cols = []

    for c in feature_cols:
        if c not in df.columns:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    return numeric_cols, categorical_cols


def build_preprocessor(df, feature_cols):
    numeric_cols, categorical_cols = infer_feature_types(df, feature_cols)

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop"
    )
    return preprocessor, numeric_cols, categorical_cols

In [9]:
def get_selection_model(target_type, random_state=42, cv=3):
    """
    Models used inside bootstrap stability selection.
    """
    if target_type == "continuous":
        # ElasticNetCV is more stable than pure Lasso when predictors are correlated
        model = ElasticNetCV(
            l1_ratio=[0.2, 0.5, 0.8, 1.0],
            cv=cv,
            random_state=random_state,
            n_alphas=50,
            max_iter=5000
        )
    elif target_type == "binary":
        model = LogisticRegressionCV(
            penalty="l1",
            solver="saga",
            cv=cv,
            scoring="roc_auc",
            random_state=random_state,
            max_iter=5000,
            n_jobs=-1
        )
    elif target_type == "categorical":
        model = LogisticRegressionCV(
        penalty="l1",
        solver="saga",
        cv=cv,
        scoring="accuracy",
        random_state=random_state,
        max_iter=5000,
        n_jobs=-1
        )
    else:
        raise ValueError(f"Unsupported target_type: {target_type}")
    return model

In [10]:
def get_nonzero_features_from_fitted_model(model, feature_names, target_type):
    """
    Extract selected features after fitting.
    """
    if target_type == "continuous":
        coefs = np.asarray(model.coef_).ravel()
        selected = [f for f, c in zip(feature_names, coefs) if np.abs(c) > 1e-8]
        return selected

    elif target_type == "binary":
        coefs = np.asarray(model.coef_).ravel()
        selected = [f for f, c in zip(feature_names, coefs) if np.abs(c) > 1e-8]
        return selected

    elif target_type == "categorical":
        coefs = np.asarray(model.coef_)
        selected_mask = (np.abs(coefs) > 1e-8).any(axis=0)
        selected = [f for f, keep in zip(feature_names, selected_mask) if keep]
        return selected

    else:
        raise ValueError(f"Unsupported target_type: {target_type}")

In [11]:
def get_processed_feature_names(preprocessor, numeric_cols, categorical_cols):
    """
    For this implementation:
    - numeric columns remain numeric_cols
    - categorical columns remain categorical_cols because we are only imputing, not one-hot encoding
    """
    feature_names = []

    # numeric
    feature_names.extend(numeric_cols)

    # categorical (after one-hot)
    if len(categorical_cols) > 0:
        ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        cat_names = ohe.get_feature_names_out(categorical_cols)
        feature_names.extend(cat_names.tolist())

    return feature_names

In [12]:
def stability_select_predictors(
    df,
    target,
    feature_cols,
    target_type,
    n_bootstrap=50,
    subsample_frac=0.7,
    selection_threshold=0.6,
    random_state=42
):
    """
    Returns:
    - stable_selected: list of selected predictors
    - selection_freq: pd.Series with selection frequencies
    """

    work = df[[target] + feature_cols].copy()

    # Drop rows where target missing
    work = work[~work[target].isna()].copy()

    # If target has too few valid rows, return empty
    if len(work) < 50:
        return [], pd.Series(dtype=float)

    # Encode categorical/binary target if needed
    y_raw = work[target].copy()
    label_encoder = None

    if target_type in ["binary", "categorical"]:
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(y_raw.astype(str))
    else:
        y = pd.to_numeric(y_raw, errors="coerce")
        mask = ~pd.isna(y)
        work = work.loc[mask].copy()
        y = y.loc[mask].values

    X = work[feature_cols].copy()

    preprocessor, numeric_cols, categorical_cols = build_preprocessor(X, feature_cols)
    X_proc = preprocessor.fit_transform(X)
    processed_feature_names = get_processed_feature_names(preprocessor, numeric_cols, categorical_cols)

    n = X_proc.shape[0]
    rng = np.random.RandomState(random_state)

    selection_counts = pd.Series(0, index=processed_feature_names, dtype=float)

    for b in range(n_bootstrap):
        idx = rng.choice(np.arange(n), size=int(subsample_frac * n), replace=False)
        X_b = X_proc[idx]
        y_b = y[idx]

        # skip degenerate bootstrap samples
        if target_type in ["binary", "categorical"]:
            if len(np.unique(y_b)) < 2:
                continue

        model = get_selection_model(target_type, random_state=random_state + b)
        model.fit(X_b, y_b)

        selected = get_nonzero_features_from_fitted_model(model, processed_feature_names, target_type)
        selection_counts.loc[selected] += 1

    selection_freq = selection_counts / n_bootstrap
    stable_selected = selection_freq[selection_freq >= selection_threshold].index.tolist()

    return stable_selected, selection_freq.sort_values(ascending=False)

In [13]:
def collapse_onehot_features(selected_features):
    base_vars = set()
    for f in selected_features:
        if "_" in f:
            base = f.split("_")[0]
            base_vars.add(base)
        else:
            base_vars.add(f)
    return list(base_vars)

In [14]:
def apply_forced_predictors(target, selected_predictors, all_allowed_predictors, time_col, max_lag,
                            force_include_time=True, force_include_own_lag=True):
    out = list(selected_predictors)

    if force_include_time and time_col in all_allowed_predictors and time_col not in out:
        out.append(time_col)

    own_lag = f"lag1_{target}"
    if force_include_own_lag and max_lag >= 1 and own_lag in all_allowed_predictors and own_lag not in out:
        out.append(own_lag)

    return list(dict.fromkeys(out))

In [15]:
def refit_simple_model(df, target, selected_predictors, target_type):
    """
    Refit simple unpenalized model on selected predictors.
    Returns:
    - fitted model bundle
    - performance summary
    """
    if len(selected_predictors) == 0:
        return None, {"status": "no_predictors"}

    work = df[[target] + selected_predictors].copy()
    work = work[~work[target].isna()].copy()

    if len(work) < 50:
        return None, {"status": "too_few_rows"}

    y_raw = work[target].copy()
    X = work[selected_predictors].copy()

    preprocessor, numeric_cols, categorical_cols = build_preprocessor(X, selected_predictors)
    X_proc = preprocessor.fit_transform(X)
    feature_names = get_processed_feature_names(preprocessor, numeric_cols, categorical_cols)

    # encode targets if needed
    label_encoder = None

    if target_type == "continuous":
        y = pd.to_numeric(y_raw, errors="coerce")
        mask = ~pd.isna(y)
        X_proc = X_proc[mask.values]
        y = y.loc[mask].values
        model = LinearRegression()
        model.fit(X_proc, y)
        yhat = model.predict(X_proc)
        perf = {
            "status": "ok",
            "n": len(y),
            "metric": "r2",
            "value": float(r2_score(y, yhat))
        }

    elif target_type == "binary":
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(y_raw.astype(str))
        if len(np.unique(y)) < 2:
            return None, {"status": "single_class"}
        model = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=5000
        )
        model.fit(X_proc, y)
        yprob = model.predict_proba(X_proc)[:, 1]
        perf = {
            "status": "ok",
            "n": len(y),
            "metric": "auc",
            "value": float(roc_auc_score(y, yprob))
        }

    elif target_type == "categorical":
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(y_raw.astype(str))
        if len(np.unique(y)) < 2:
            return None, {"status": "single_class"}
        model = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=5000
        )
        model.fit(X_proc, y)
        yhat = model.predict(X_proc)
        perf = {
            "status": "ok",
            "n": len(y),
            "metric": "accuracy",
            "value": float(accuracy_score(y, yhat))
        }

    else:
        raise ValueError(f"Unsupported target_type: {target_type}")

    bundle = {
        "preprocessor": preprocessor,
        "feature_names": feature_names,
        "model": model,
        "label_encoder": label_encoder,
        "target_type": target_type
    }
    return bundle, perf

In [16]:
def make_signature(target, predictors):
    if len(predictors) == 0:
        return f"{target} ~ 1"
    return f"{target} ~ " + " + ".join(predictors)

In [17]:
def discover_covmodel_signatures(
    df,
    covnames,
    baseline_cols,
    id_col,
    time_col,
    block_index_map,
    target_type_map,
    max_lag=1,
    n_bootstrap=50,
    subsample_frac=0.7,
    selection_threshold=0.6,
    force_include_time=True,
    force_include_own_lag=True,
    random_state=42
):
    df_work = df.copy()

    # create lag features
    df_work = add_lag_features(df_work, id_col=id_col, time_col=time_col, variables=covnames, max_lag=max_lag)

    results = []
    signature_map = OrderedDict()
    selection_freqs = {}

    for target in covnames:
        print(f"\n=== Processing: {target} ===")

        target_type = target_type_map[target]

        allowed_predictors = get_allowed_predictors(
            target=target,
            covnames=covnames,
            baseline_cols=baseline_cols,
            time_col=time_col,
            max_lag=max_lag,
            block_index_map=block_index_map,
            force_include_time=force_include_time,
            force_include_own_lag=force_include_own_lag
        )

        # keep only columns that actually exist
        allowed_predictors = [c for c in allowed_predictors if c in df_work.columns]

        selected_predictors, selection_freq = stability_select_predictors(
            df=df_work,
            target=target,
            feature_cols=allowed_predictors,
            target_type=target_type,
            n_bootstrap=n_bootstrap,
            subsample_frac=subsample_frac,
            selection_threshold=selection_threshold,
            random_state=random_state
        )

        selected_predictors = collapse_onehot_features(selected_predictors)

        selected_predictors = apply_forced_predictors(
            target=target,
            selected_predictors=selected_predictors,
            all_allowed_predictors=allowed_predictors,
            time_col=time_col,
            max_lag=max_lag,
            force_include_time=force_include_time,
            force_include_own_lag=force_include_own_lag
        )

        # optional: preserve original allowed-predictor ordering in final signature
        selected_predictors = [c for c in allowed_predictors if c in selected_predictors]

        bundle, perf = refit_simple_model(
            df=df_work,
            target=target,
            selected_predictors=selected_predictors,
            target_type=target_type
        )

        signature = make_signature(target, selected_predictors)
        signature_map[target] = signature
        selection_freqs[target] = selection_freq

        result_row = {
            "target": target,
            "target_type": target_type,
            "n_allowed_predictors": len(allowed_predictors),
            "n_selected_predictors": len(selected_predictors),
            "selected_predictors": selected_predictors,
            "signature": signature,
            "refit_status": perf.get("status"),
            "refit_metric": perf.get("metric"),
            "refit_value": perf.get("value"),
            "n_refit": perf.get("n")
        }
        results.append(result_row)

        print(signature)
        if perf.get("status") == "ok":
            print(f"Refit {perf['metric']}: {perf['value']:.4f}")
        else:
            print(f"Refit status: {perf.get('status')}")

    results_df = pd.DataFrame(results)
    return df_work, results_df, signature_map, selection_freqs

In [18]:
# mimicdata
mimicdata = pd.read_parquet("gdata.parquet")

# Sub sample of admissions
rng = np.random.default_rng(34)
unique_ids = mimicdata['admission_id'].dropna().unique()
sampled_ids = rng.choice(unique_ids, size=500, replace=False)
mimicdata_sub = mimicdata[mimicdata['admission_id'].isin(sampled_ids)].copy()
mimicdata_sub.reset_index(drop=True, inplace=True)

# Reduce time horizon
mimicdata_sub = mimicdata_sub[mimicdata_sub[time_col] <= 10]   # e.g. first 5 days

n_bootstrap = 10        # instead of 50
selection_threshold = 0.5

In [19]:
df = mimicdata_sub.copy()

df_lagged, results_df, signature_map, selection_freqs = discover_covmodel_signatures(
    df=df,
    covnames=covnames,
    baseline_cols=baseline_cols,
    id_col=id_col,
    time_col=time_col,
    block_index_map=block_index_map,
    target_type_map=target_type_map,
    max_lag=max_lag,
    n_bootstrap=n_bootstrap,
    subsample_frac=subsample_frac,
    selection_threshold=selection_threshold,
    force_include_time=force_include_time,
    force_include_own_lag=force_include_own_lag,
    random_state=random_state
)


=== Processing: vent_mode__last__last_12h ===
vent_mode__last__last_12h ~ age + sex + origin + t0 + lag1_vent_mode__last__last_12h
Refit accuracy: 0.9334

=== Processing: vent_mode__hours_since_last__last_12h ===
vent_mode__hours_since_last__last_12h ~ sex + origin + t0 + lag1_vent_mode__hours_since_last__last_12h
Refit r2: 0.7368

=== Processing: pco2_arterial__mean__last_12h ===
pco2_arterial__mean__last_12h ~ t0 + lag1_pco2_arterial__mean__last_12h
Refit r2: 0.7637

=== Processing: po2_arterial__mean__last_12h ===
po2_arterial__mean__last_12h ~ age + sex + t0 + lag1_po2_arterial__mean__last_12h
Refit r2: 0.6185

=== Processing: fio2__last__last_12h ===
fio2__last__last_12h ~ t0 + lag1_fio2__last__last_12h
Refit accuracy: 0.8974

=== Processing: o2_saturation__mean__last_12h ===
o2_saturation__mean__last_12h ~ age + sex + origin + t0 + lag1_o2_saturation__mean__last_12h
Refit r2: 0.4420

=== Processing: respiratory_rate_measured__mean__last_12h ===
respiratory_rate_measured__mean__l

In [25]:
results_df[[
    "target",
    "target_type",
    "n_allowed_predictors",
    "n_selected_predictors",
    "refit_metric",
    "refit_value",
    "signature"
]]

,target,target_type,n_allowed_predictors,n_selected_predictors,refit_metric,refit_value,signature
0,vent_mode__last__last_12h,categorical,23,5,accuracy,0.933378,vent_mode__last__last_12h ~ age + sex + origin...
1,vent_mode__hours_since_last__last_12h,continuous,23,4,r2,0.736808,vent_mode__hours_since_last__last_12h ~ sex + ...
2,pco2_arterial__mean__last_12h,continuous,26,2,r2,0.763749,pco2_arterial__mean__last_12h ~ t0 + lag1_pco2...
3,po2_arterial__mean__last_12h,continuous,26,4,r2,0.618509,po2_arterial__mean__last_12h ~ age + sex + t0 ...
4,fio2__last__last_12h,categorical,23,2,accuracy,0.897376,fio2__last__last_12h ~ t0 + lag1_fio2__last__l...
5,o2_saturation__mean__last_12h,continuous,26,5,r2,0.441995,o2_saturation__mean__last_12h ~ age + sex + or...
6,respiratory_rate_measured__mean__last_12h,continuous,26,2,r2,0.549809,respiratory_rate_measured__mean__last_12h ~ t0...
7,glasgow_coma_scale_total__last__last_12h,categorical,30,2,accuracy,0.906797,glasgow_coma_scale_total__last__last_12h ~ t0 ...
8,lactate__last__last_12h,continuous,31,5,r2,0.814248,lactate__last__last_12h ~ age + sex + origin +...
9,fluid_out_urine__mean__last_12h,continuous,35,5,r2,0.609319,fluid_out_urine__mean__last_12h ~ age + sex + ...


In [21]:
for target, sig in signature_map.items():
    print(sig)

vent_mode__last__last_12h ~ age + sex + origin + t0 + lag1_vent_mode__last__last_12h
vent_mode__hours_since_last__last_12h ~ sex + origin + t0 + lag1_vent_mode__hours_since_last__last_12h
pco2_arterial__mean__last_12h ~ t0 + lag1_pco2_arterial__mean__last_12h
po2_arterial__mean__last_12h ~ age + sex + t0 + lag1_po2_arterial__mean__last_12h
fio2__last__last_12h ~ t0 + lag1_fio2__last__last_12h
o2_saturation__mean__last_12h ~ age + sex + origin + t0 + lag1_o2_saturation__mean__last_12h
respiratory_rate_measured__mean__last_12h ~ t0 + lag1_respiratory_rate_measured__mean__last_12h
glasgow_coma_scale_total__last__last_12h ~ t0 + lag1_glasgow_coma_scale_total__last__last_12h
lactate__last__last_12h ~ age + sex + origin + t0 + lag1_lactate__last__last_12h
fluid_out_urine__mean__last_12h ~ age + sex + origin + t0 + lag1_fluid_out_urine__mean__last_12h
ureum__last__last_12h ~ age + sex + origin + t0 + lag1_ureum__last__last_12h
creatinine__last__last_12h ~ origin + t0 + lag1_creatinine__last__

In [22]:
covmodels = [signature_map[target] for target in covnames]
covmodels

['vent_mode__last__last_12h ~ age + sex + origin + t0 + lag1_vent_mode__last__last_12h',
 'vent_mode__hours_since_last__last_12h ~ sex + origin + t0 + lag1_vent_mode__hours_since_last__last_12h',
 'pco2_arterial__mean__last_12h ~ t0 + lag1_pco2_arterial__mean__last_12h',
 'po2_arterial__mean__last_12h ~ age + sex + t0 + lag1_po2_arterial__mean__last_12h',
 'fio2__last__last_12h ~ t0 + lag1_fio2__last__last_12h',
 'o2_saturation__mean__last_12h ~ age + sex + origin + t0 + lag1_o2_saturation__mean__last_12h',
 'respiratory_rate_measured__mean__last_12h ~ t0 + lag1_respiratory_rate_measured__mean__last_12h',
 'glasgow_coma_scale_total__last__last_12h ~ t0 + lag1_glasgow_coma_scale_total__last__last_12h',
 'lactate__last__last_12h ~ age + sex + origin + t0 + lag1_lactate__last__last_12h',
 'fluid_out_urine__mean__last_12h ~ age + sex + origin + t0 + lag1_fluid_out_urine__mean__last_12h',
 'ureum__last__last_12h ~ age + sex + origin + t0 + lag1_ureum__last__last_12h',
 'creatinine__last__la

In [23]:
target = "A"   # change as needed
selection_freqs[target].head(30)

age                                                           1.0
sex                                                           1.0
t0                                                            1.0
lag1_vent_mode__hours_since_last__last_12h                    1.0
lag1_pco2_arterial__mean__last_12h                            1.0
lag1_o2_saturation__mean__last_12h                            1.0
lag1_respiratory_rate_measured__mean__last_12h                1.0
lag1_lactate__last__last_12h                                  1.0
lag1_hemoglobin__last__last_12h                               1.0
lag1_ureum__last__last_12h                                    1.0
lag1_creatinine__last__last_12h                               1.0
lag1_arterial_blood_pressure_mean__mean__last_12h             1.0
lag1_heart_rate__mean__last_12h                               1.0
o2_saturation__mean__last_12h                                 1.0
vent_mode__hours_since_last__last_12h                         1.0
lag1_activ

In [24]:
#Strong recommendation: use mostly lags for physiology
#This code already allows same-time predictors only from earlier blocks. That is deliberate. If you want to be more conservative, remove same-time predictors entirely except for a few process variables.
#To do that, replace this line in get_allowed_predictors:

predictors.extend(get_allowed_same_time_predictors(target, covnames, block_index_map))

NameError: name 'predictors' is not defined

In [ ]:
#A more conservative version I would personally use for your project
#For ICU physiology, I would usually make the allowed set:

#baseline cols
#time
#own lag
#lags of all other covariates
#very few same-time predictors

#because that is safer for simulation.

#If you want that version, change get_allowed_predictors to:

def get_allowed_predictors(
    target,
    covnames,
    baseline_cols,
    time_col,
    max_lag,
    block_index_map,
    force_include_time=True,
    force_include_own_lag=True
):
    predictors = []
    predictors.extend(baseline_cols)

    if force_include_time:
        predictors.append(time_col)

    for lag in range(1, max_lag + 1):
        predictors.extend([f"lag{lag}_{v}" for v in covnames])

    predictors = [p for p in predictors if p != target]
    predictors = list(dict.fromkeys(predictors))

    if force_include_own_lag:
        own_lag = f"lag1_{target}"
        if own_lag not in predictors and max_lag >= 1:
            predictors.append(own_lag)

    return predictors